# ECABSD V2 — Baseline Recovery Training
This notebook recovers the missing enriched graphs (x=33, edge=5) and trains the V2 baseline.

In [ ]:
# ============================================================
# CELL 1: Wipe old code and clone clean repo
# ============================================================
import os, shutil

WORK = '/kaggle/working'
REPO = 'https://github.com/VigneshReddyKura/ecabsd.git'
DEST = f'{WORK}/ecabsd'

os.chdir(WORK)

if os.path.exists(DEST):
    shutil.rmtree(DEST)
    print('[SETUP] Removed old ghost code.')

print('Cloning...')
!git clone {REPO} {DEST}

if not os.path.exists(os.path.join(DEST, 'train.py')):
    raise RuntimeError('Clone failed! train.py not found.')

os.chdir(DEST)

print('PWD:', os.getcwd())
print('Repo files:', sorted(os.listdir('.')))
print('✅ Repo cloned successfully')

In [ ]:
# ============================================================
# CELL 2: Install dependencies
# ============================================================
import subprocess, sys, torch

def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

print('[DEPS] Installing basic packages...')
pip_install(['biopython', 'scikit-learn', 'pandas', 'transformers'])

print('[DEPS] Installing PyG & pydssp (can take 2 mins)...')
pip_install(['torch_geometric'])
pip_install(['pydssp'])

print('[DEPS] ✅ All dependencies installed.')

In [ ]:
# ============================================================
# CELL 3: Find and link dataset
# ============================================================
import os, shutil

input_base = '/kaggle/input'
ds_dir = None

for root, dirs, files in os.walk(input_base):
    if 'processed' in dirs and 'splits.csv' in files:
        ds_dir = root
        break

if not ds_dir:
    raise RuntimeError('Dataset not found! Please attach it to the notebook.')

print(f'[DATA] Found dataset at: {ds_dir}')

if os.path.exists('data/processed'):
    shutil.rmtree('data/processed')
os.makedirs('data/processed', exist_ok=True)

shutil.copytree(os.path.join(ds_dir, 'processed'), 'data/processed', dirs_exist_ok=True)
shutil.copy2(os.path.join(ds_dir, 'splits.csv'), 'data/splits.csv')
print('[DATA] ✅ Data copied into working dir.')

In [ ]:
# ============================================================
# CELL 4: Fix data leakage
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/splits.csv')
unique_pdbs = df['pdb_id'].unique()

train_ids, temp_ids = train_test_split(unique_pdbs, test_size=0.3, random_state=42)
val_ids, test_ids   = train_test_split(temp_ids, test_size=0.5, random_state=42)

df.loc[df['pdb_id'].isin(train_ids), 'split'] = 'train'
df.loc[df['pdb_id'].isin(val_ids), 'split']   = 'val'
df.loc[df['pdb_id'].isin(test_ids), 'split']  = 'test'

df.to_csv('data/splits.csv', index=False)
print('[LEAKAGE] ✅ Splits rebuilt with strict zero-overlap.')

In [ ]:
# ============================================================
# CELL 5: Remove bad graphs (x != 33 or edge != 5)
# ============================================================
import torch, glob, os, shutil

SRC = 'data/processed'
BAD = 'data/bad_graphs'
os.makedirs(BAD, exist_ok=True)

good = 0
bad  = 0

for f in glob.glob(SRC + '/*.pt'):
    try:
        g = torch.load(f, map_location='cpu', weights_only=False)
        x_ok = hasattr(g, 'x') and g.x is not None and g.x.dim() == 2 and g.x.shape[1] == 33
        e_ok = hasattr(g, 'edge_attr') and g.edge_attr is not None and g.edge_attr.dim() == 2 and g.edge_attr.shape[1] == 5
        if x_ok and e_ok:
            good += 1
        else:
            bad += 1
            shutil.move(f, os.path.join(BAD, os.path.basename(f)))
    except Exception as e:
        bad += 1
        shutil.move(f, os.path.join(BAD, os.path.basename(f)))

print('Good graphs:', good)
print('Moved bad graphs:', bad)
print('Final graphs in processed:', len(glob.glob(SRC + '/*.pt')))
print('Ready for recovery step.')

In [ ]:
# ============================================================
# CELL 6: Auto-Recover Bad Graphs
# Downloads raw PDBs and rebuilds missing/bad graphs with x=33
# ============================================================
import sys, subprocess
print('[RECOVERY] Starting graph recovery script...')
subprocess.run([sys.executable, 'scripts/recover_graphs.py'])
print('[RECOVERY] Done.')

In [ ]:
# ============================================================
# CELL 7: Update config.yaml with final dimensions + verify
# ============================================================
import yaml, os, pandas as pd

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['processed_dir']      = 'data/processed'
cfg['data']['splits_csv']         = 'data/splits.csv'
cfg['model']['esm_dim']           = 33      # match actual node feature dim
cfg['model']['edge_feature_dim']  = 5       # match actual edge feature dim
cfg['training']['epochs']         = 100
cfg['training']['num_workers']    = 0

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

final_cfg = yaml.safe_load(open('config.yaml'))
df = pd.read_csv(final_cfg['data']['splits_csv'])
vc = df['split'].value_counts()
print('Usable sample counts:')
print(f"  train : {vc.get('train', 0)}")
print(f"  val   : {vc.get('val',   0)}")
print(f"  test  : {vc.get('test',  0)}")

print('\nAll checks passed — safe to train!')

In [ ]:
# ============================================================
# CELL 8: TRAIN CELL — restore repo data code, verify paths, then train
# ============================================================
import os, sys, subprocess, yaml, glob

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

print('[TRAIN] Current dir:', os.getcwd())

# 1. Restore repo Python files if dataset overwrote data/
print('[TRAIN] Restoring repo data module...')
subprocess.run(['git', 'checkout', '--', 'data/dataset.py', 'data/__init__.py'], cwd=WORKDIR)

# 2. Verify data module exists
print('[TRAIN] data folder:', os.listdir('data')[:20])

if not os.path.exists('data/dataset.py'):
    raise FileNotFoundError('data/dataset.py still missing. Repo data module not restored.')

# 3. Fix config paths
with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['processed_dir'] = 'data/processed'
cfg['data']['splits_csv'] = 'data/splits.csv'

with open('config.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

print('[TRAIN] Config data section:')
print(cfg['data'])

# 4. Verify graphs/splits
print('[TRAIN] Graphs:', len(glob.glob('data/processed/*.pt')))
print('[TRAIN] Splits exists:', os.path.exists('data/splits.csv'))

if len(glob.glob('data/processed/*.pt')) == 0:
    raise RuntimeError('No .pt graphs found in data/processed')

if not os.path.exists('data/splits.csv'):
    raise RuntimeError('data/splits.csv missing')

# 5. Run training
print('[TRAIN] Starting training...')
result = subprocess.run([sys.executable, 'train.py'], cwd=WORKDIR)

if result.returncode != 0:
    raise RuntimeError('[TRAIN] Training failed. Check error above.')
else:
    print('[TRAIN] ✅ Training completed successfully!')

In [ ]:
# ============================================================
# CELL 9: Evaluate on test set
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, 'evaluate.py'])

In [ ]:
# ============================================================
# CELL 10: Verify timestamp + package everything for download
# ============================================================
import os, time, zipfile

ckpt = 'checkpoints/best_model.pt'
if os.path.exists(ckpt):
    mt = os.path.getmtime(ckpt)
    sz = os.path.getsize(ckpt)
    print('Checkpoint verification:')
    print(f'  best_model.pt    : {time.ctime(mt)}  ({sz//1024} KB)')
else:
    print('❌ ERROR: best_model.pt not found! Training failed.')

zip_path = '/kaggle/working/ecabsd_v2_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk('checkpoints'):
        for f in files:
            zf.write(os.path.join(root, f))
    for root, _, files in os.walk('results'):
        for f in files:
            zf.write(os.path.join(root, f))
    for root, _, files in os.walk('models'):
        for f in files:
            if f.endswith('.py'):
                zf.write(os.path.join(root, f))

print(f'\n✅ Download ready: {zip_path}')
print('Extract and copy models/ + checkpoints/ into your local ecabsd folder.')